## 1. Loading the data

In [69]:
import pandas as pd
import numpy as np

player_roles1 = pd.read_parquet("data_raw/player_roles1.parquet")
player_roles2 = pd.read_parquet("data_raw/player_roles2.parquet")

trajectories1 = pd.read_parquet("data_raw/trajectories1.parquet")
trajectories2 = pd.read_parquet("data_raw/trajectories2.parquet")



## 2. Merging data files and breaking up trajectories

### 2.1 Merging Trajectories

In [70]:
trajectories = pd.concat([trajectories1,trajectories2], axis = 0, ignore_index = True)
print(f"Lenght of data: {trajectories.shape}")
print(trajectories.head())

Lenght of data: (482260, 4)
                                               puuid         match_id  team  \
0  mCG4W3ohaS1yeYuEIpAiXCgHPyOb_GsrpP_Yw6y7Y-88SI...  EUW1_7684872966     0   
1  QfvaTyT_3Ez0jaMf0_QzLaOqAwRGPKGg179xIKLtNoFoof...  EUW1_7684872966     0   
2  nJgPqmoqt_bWlQgnH9U3XDORpE9zbhgTz8HzEXRChAmH8u...  EUW1_7684872966     0   
3  D29BAybAOJFm1tgl1K_r9BOC_vHCgBNDPH6Bnjl628rDbh...  EUW1_7684872966     0   
4  3XpUlB8N3-_zm4qB0JykZOtTsETI3GBCxXKe60EAUDNYrW...  EUW1_7684872966     0   

                                           positions  
0  [[603, 611], [1032, 12131], [1401, 11087], [82...  
1  [[662, 285], [3101, 8063], [6883, 5349], [1059...  
2  [[362, 135], [7572, 7012], [7244, 7647], [8182...  
3  [[130, 401], [10633, 1241], [10709, 1448], [11...  
4  [[298, 675], [11464, 1854], [10734, 1746], [11...  


### 2.2 Merging Roles

In [71]:
player_roles = pd.concat([player_roles1,player_roles2], axis = 0, ignore_index = True)
print(f"Lenght of data: {player_roles.shape}")
print(player_roles.head())

Lenght of data: (487950, 4)
                                               puuid         match_id  \
0  F77cxzk_zBZefmsFG-XopOhlcobFU0xIgTLxTYbwJ4TYAb...  EUW1_7682224814   
1  Elayv1cr5-cfcVkACT6tZmuDloCiNJJ1TVXxBIr6rUE0w7...  EUW1_7682224814   
2  3MuZ03offdBavQ8mRk4iAeQeDC1s25d-2cbcIQ5ceZ4vFN...  EUW1_7682224814   
3  HmSVRNlLrEwGC9ujRkJcGyskPkBqx6ahqogJbIIvZd7ZET...  EUW1_7682224814   
4  GKYhuj2C0zbsQcEpEhlw9d5kBGcPo0U5O2TL4-Atvvt13E...  EUW1_7682224814   

      role champion  
0      TOP    Fiora  
1   JUNGLE   Graves  
2   MIDDLE     Azir  
3   BOTTOM   Ezreal  
4  UTILITY     Rell  


### 2.3 Merging Trajectories and Roles on puuid and match_id

In [72]:
merged = trajectories.merge(
    player_roles,
    on=["puuid", "match_id"],
    how="left"
)
print(f"Lenght of data: {len(merged)}")
print(merged.head())

Lenght of data: 482260
                                               puuid         match_id  team  \
0  mCG4W3ohaS1yeYuEIpAiXCgHPyOb_GsrpP_Yw6y7Y-88SI...  EUW1_7684872966     0   
1  QfvaTyT_3Ez0jaMf0_QzLaOqAwRGPKGg179xIKLtNoFoof...  EUW1_7684872966     0   
2  nJgPqmoqt_bWlQgnH9U3XDORpE9zbhgTz8HzEXRChAmH8u...  EUW1_7684872966     0   
3  D29BAybAOJFm1tgl1K_r9BOC_vHCgBNDPH6Bnjl628rDbh...  EUW1_7684872966     0   
4  3XpUlB8N3-_zm4qB0JykZOtTsETI3GBCxXKe60EAUDNYrW...  EUW1_7684872966     0   

                                           positions     role champion  
0  [[603, 611], [1032, 12131], [1401, 11087], [82...      TOP     Ornn  
1  [[662, 285], [3101, 8063], [6883, 5349], [1059...   JUNGLE    Diana  
2  [[362, 135], [7572, 7012], [7244, 7647], [8182...   MIDDLE  Taliyah  
3  [[130, 401], [10633, 1241], [10709, 1448], [11...   BOTTOM    Yasuo  
4  [[298, 675], [11464, 1854], [10734, 1746], [11...  UTILITY    Leona  


### 2.4 Flattening and breaking up trajectories

In [73]:
merged["traj_len"] = merged["positions"].apply(len)

max_len = merged["traj_len"].max()
print("Longest game length:", max_len)

Longest game length: 60


In [74]:
def normalize_positions(pos):
    """
    Converts: array([array([x,y]), array([x,y]), ...], dtype=object) -> array([[x,y], [x,y], ...])
    """
    return np.vstack(pos)

merged["positions_xy"] = merged["positions"].apply(normalize_positions)


def expand_positions(pos_list, max_steps):
    out = np.full((max_steps, 2), np.nan)
    n = min(len(pos_list), max_steps)
    out[:n] = pos_list[:n]
    return out.flatten()

expanded = np.vstack(
    merged["positions_xy"].apply(lambda p: expand_positions(p, max_len))
)

columns = []
for i in range(max_len):
    columns.extend([f"x{i}", f"y{i}"])

positions_df = pd.DataFrame(expanded, columns=columns)

final_df = pd.concat(
    [merged.drop(columns=["positions","positions_xy"]), positions_df],
    axis=1
)
print(final_df[["x0","y0","x1","y1"]])

             x0       y0       x1       y1
0         603.0    611.0   1032.0  12131.0
1         662.0    285.0   3101.0   8063.0
2         362.0    135.0   7572.0   7012.0
3         130.0    401.0  10633.0   1241.0
4         298.0    675.0  11464.0   1854.0
...         ...      ...      ...      ...
482255  14103.0  14195.0   1678.0  12860.0
482256  14426.0  14170.0   7261.0  11357.0
482257  14321.0  14673.0   7590.0   7865.0
482258  14589.0  14454.0  13054.0   3610.0
482259  14055.0  14493.0  13526.0   3548.0

[482260 rows x 4 columns]


## 3. Feature Engineering

## 4. Data errors, missing labels and outliers

### 4.1 Searching for errors/NaN in labels

We are looking for:
- NaN roles
- Not assigned roles which are empty but not NaN
- Duplicated which might have been created during merging

(if you have any more ideas, write them here and implement them)

In [75]:
VALID_ROLES = {"TOP", "JUNGLE", "MIDDLE", "BOTTOM", "UTILITY"}
invalid_roles = final_df[~final_df["role"].isin(VALID_ROLES)]


print(f"Count of nan labels: {final_df["role"].isna().sum()}")
print(f"Count of glitched roles: {len(invalid_roles)}")
#checks to see if there are any players with 2 or more roles in a single match 
print(f"Number of duplicate errors: {player_roles.duplicated(subset=["puuid", "match_id"]).sum()}")

Count of nan labels: 0
Count of glitched roles: 72
Number of duplicate errors: 0


We can either:

1. Delete them

   **Pros:**
   - No errors
   - We might need it later for some analysis (not sure)

   **Cons:**
   - Losing data (but it’s only 72 rows out of 480k)

In [76]:
data_deleted = final_df.drop(invalid_roles.index)
print(data_deleted.shape)

(482188, 126)


2. Fill the role based on the majority of roles the champion is played on 

   **Pros:** 
   - Really good for some champions like kha'zix, which are played on one role only. 
   
   **Cons:**
   - Bad for champions which are played frequently on 2 roles or more.

In [77]:
champion_role_prior = (
    final_df[final_df["role"].isin(VALID_ROLES)]
    .groupby(["champion", "role"])
    .size()
    .reset_index(name="count")
    .sort_values(["champion", "count"], ascending=[True, False])
    .drop_duplicates("champion")
    .set_index("champion")["role"]
)


invalid_roles.loc[:,["role_champion_majority"]] = (
    invalid_roles["champion"].map(champion_role_prior)
)

print(invalid_roles.loc[:,["puuid", "match_id","champion","role_champion_majority"]].head())

                                                   puuid         match_id  \
2441   S-PwB_dwIj_BLVb9OCp9VrD9_A_-6gB9r0UrHeKGmMuTJK...   NA1_5462008265   
2798   IFykstMttbxyx1lKPlpiROJijBc0ZDIoVfJVoW8uD91DQc...  EUW1_7684088114   
9304   12p2wtYP2BN5lsJepD5MCh4GpZYP5GB6knYdnGOXv7vXX_...    KR_8027299858   
33810  s5xnoUd1sI5tpwn80u_Jgkflqqp_E-i3ie2csj-FVEFKre...  EUW1_7684307377   
38048  Yahx_bxKheeyo1MNBUVWvTI-de5QS9A8FB5kt8dS08fa2R...  EUW1_7684953000   

       champion role_champion_majority  
2441     Khazix                 JUNGLE  
2798   Tristana                 BOTTOM  
9304       Bard                UTILITY  
33810      Sett                    TOP  
38048    Twitch                 BOTTOM  


In [78]:
data_majority = final_df.copy()
data_majority.loc[
    invalid_roles.index,
    "role"
] = invalid_roles["role_champion_majority"].values

3. Fill the role by looking which Role isnt in the team 

   **Pros:**
   - 100% accurate when only one role from a given team in a match is missing.
   
   **Cons:** 
   - Might backfire if there is more than 1 role missing in on match.
   - Only works if riot's predictions are correct.

In [79]:
def infer_missing_role(match_id, team):
    roles_present = team_roles.get((match_id, team), set())
    missing = VALID_ROLES - roles_present
    return next(iter(missing)) if len(missing) == 1 else None

team_roles = (
    final_df[final_df["role"].isin(VALID_ROLES)]
    .groupby(["match_id", "team"])["role"]
    .apply(set)
)

invalid_roles.loc[:,"role_team_missing"] = invalid_roles.apply(
    lambda row: infer_missing_role(row["match_id"], row["team"]),
    axis=1
)

print(invalid_roles.loc[:,["puuid", "match_id","champion","role_team_missing"]].head())

                                                   puuid         match_id  \
2441   S-PwB_dwIj_BLVb9OCp9VrD9_A_-6gB9r0UrHeKGmMuTJK...   NA1_5462008265   
2798   IFykstMttbxyx1lKPlpiROJijBc0ZDIoVfJVoW8uD91DQc...  EUW1_7684088114   
9304   12p2wtYP2BN5lsJepD5MCh4GpZYP5GB6knYdnGOXv7vXX_...    KR_8027299858   
33810  s5xnoUd1sI5tpwn80u_Jgkflqqp_E-i3ie2csj-FVEFKre...  EUW1_7684307377   
38048  Yahx_bxKheeyo1MNBUVWvTI-de5QS9A8FB5kt8dS08fa2R...  EUW1_7684953000   

       champion role_team_missing  
2441     Khazix            JUNGLE  
2798   Tristana            BOTTOM  
9304       Bard           UTILITY  
33810      Sett               TOP  
38048    Twitch            BOTTOM  


C:\Users\kacpe\AppData\Local\Temp\ipykernel_7216\2117583225.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  invalid_roles.loc[:,"role_team_missing"] = invalid_roles.apply(


In [80]:
data_team = final_df.copy()
data_team.loc[
    invalid_roles.index,
    "role"
] = invalid_roles["role_team_missing"].values

### 4.2 Comapring method 2 and 3

In [81]:
print(f"Roles filled by majority voting: {invalid_roles["role_champion_majority"].notna().sum()}")
print(f"Roles filled by Team composition analisys: {invalid_roles["role_team_missing"].notna().sum()}")

disagreement = invalid_roles[
    invalid_roles["role_champion_majority"] !=
    invalid_roles["role_team_missing"]
]

print(f"Number of confliciting fillings: {len(disagreement)}")
print(disagreement.loc[:,["puuid", "match_id","champion","role_team_missing","role_champion_majority"]])

Roles filled by majority voting: 72
Roles filled by Team composition analisys: 72
Number of confliciting fillings: 13
                                                    puuid         match_id  \
54835   oWGCPGCAlXqhbYyIH6PfC274TvAy-QWYTg3bdTPTKVwvuy...    KR_8027055351   
150209  f36h-i5xNhcWRlgqL2Ed-ZGjZNrYEbc2pBj3eZwCgR4FBf...  EUW1_7683614522   
181233  KLZz5yBwzSW66I3oO4X_ADpYjk5KZLfDf5wAMunJAi-FS-...  EUW1_7684386410   
230437  NJrYjtWmR3xnZOQXC5N2Te39xRuGo2XzUF1BbVwK8H5VSo...    KR_8027180300   
234082  ag3_xjhw9zYEjCNwMa7XeyvY4ayjf2QUhEpSIVndrUpW4R...    KR_8026846303   
241908  7vjMFW36NHqX3y_ybWUKMRCwcODgH7YbPah_L319dPcM3x...    KR_8027736569   
248362  JcNSt1i8wfKyPUsLuX4dGDQ154KVf5Pv1376LJ14P7DJkC...   NA1_5462328117   
262291  cxDHeUpd4NQrsdl0WtYBZkzyadtCMt6F3GqzovwSzGw71P...   NA1_5463749838   
297645  9fOSF6YcYaObLoHiVXGxeC79lMRuhCVwpQEt5hjK7jlNsW...  EUW1_7680078948   
300420  MpsUxQZs5EhtSPrLnIQYO22mMGJqiqTCWxIYp9JWw8VdY0...  EUW1_7684831558   
322143  zwF8yle_HTcB82dY

As i turns out Team composition analysis is a great idea, but some of the roles in the matches are **labeled incorrectly by Riot's predictor**. Which results in silly fillings such as Yunara UTILITY, as Blitzcrank was assigned BOT this game, while the champion role predictor assigns Yunara correctly as BOT.
But due to the con i talked about champions like Zed, who can be played on multiple roles, are assigned to the majority role, while it is already filled by another jungler in that match.

In [82]:
confusion_table = pd.crosstab(
    disagreement["role_team_missing"],
    disagreement["role_champion_majority"],
    rownames=["Team-missing role"],
    colnames=["Champion-majority role"]
)
confusion_table

Champion-majority role,BOTTOM,JUNGLE,MIDDLE,TOP
Team-missing role,,,,
BOTTOM,0,0,2,0
JUNGLE,0,0,0,1
MIDDLE,1,1,0,1
TOP,1,2,0,0
UTILITY,3,0,1,0


### 3.3 See how many riot labeling errors we can detect in the data(outliers)  

## x. Saving the data

In [ ]:

data_deleted.to_parquet("data_processed/data_deleted.parquet", index = False)
data_majority.to_parquet("data_processed/data_majority.parquet", index = False)
data_team.to_parquet("data_processed/data_team.parquet", index = False)